# Publish a Model: Serving in Python

**Working Example.** Copy this file, rename it, and modify *your* copy.

- Author: Denise Case, Kim Hummel
- Date: 2026-06
- Dataset: Seaborn Penguins
- Target: sex
  

## M6. Serving

This notebook trains a model, saves it, reloads it, and 
tests the **serving core** - the function a web server would call - in-process. 

It does **not** launch a blocking server inside the notebook; 
the real server is shown for you to run with `uv`. 

## Overview

This project uses the penguins dataset.
We choose to predict the target `sex`.
This target is a **discrete category**, so we have a:

- supervised ML problem (because we've chosen a target)
- a classification problem (because our target is a category)

## Section 1. Project Setup and Imports

In [8]:
# === Section 1a. DECLARE IMPORTS ===


from importlib.metadata import version  # to verify  # to verify
import logging  # for type hinting  # for type hinting
import os
from pathlib import Path
import platform  # to verify  # to verify
from typing import Any  # for type hinting  # for type hinting

from datafun_toolkit.logger import get_logger, log_header
import joblib
import pandas as pd

from mlstudio.model_builder_hummel import (
    DATASET_NAME,
    MODEL_PATH,
    TARGET_COL,
    load_data,
    save_model,
    split_data,
    summarize,
    train_model,
)

# Walk up until we find pyproject.toml (project root marker)
while not Path("pyproject.toml").exists():
    os.chdir("..")


from mlstudio.serve_hummel import predict_from_features  # noqa: E402

# Walk up until we find pyproject.toml (project root marker)
while not Path("pyproject.toml").exists():
    os.chdir("..")


# === Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK ===

LOG: logging.Logger = get_logger("M06", level="DEBUG")
log_header(LOG, "M06")


# === Section 1c. USE THE LOGGER TO VERIFY IMPORTS ===

# If any do NOT return a version number, then that package is not installed correctly.
# Check your pyproject.toml and re-run environment setup commands.

LOG.info("Confirming installation:")
LOG.info(f"  python:       {platform.python_version()}")
LOG.info(f"  pandas:       {version('pandas')}")

# === Section 1d. SET PANDAS DISPLAY CONFIGURATION (helps in notebooks) ===

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# === Section 1e. GLOBAL CONSTANTS AND CONFIGURATION ===

# CUSTOM: where the published model artifact is written.
LOG.info(f"Model artifact will be saved to: {MODEL_PATH}")

2026-08-07 20:01:31 | INFO | M06 | === RUN START ===
2026-08-07 20:01:31 | INFO | M06 | project=M06
2026-08-07 20:01:31 | INFO | M06 | repo_dir=ml-06-serving
2026-08-07 20:01:31 | INFO | M06 | python=3.14.3
2026-08-07 20:01:31 | INFO | M06 | os=Windows 11
2026-08-07 20:01:31 | INFO | M06 | shell=powershell
2026-08-07 20:01:31 | INFO | M06 | cwd=.
2026-08-07 20:01:31 | INFO | M06 | github_actions=False
2026-08-07 20:01:31 | INFO | M06 | Confirming installation:
2026-08-07 20:01:31 | INFO | M06 |   python:       3.14.3
2026-08-07 20:01:31 | INFO | M06 |   pandas:       3.0.5
2026-08-07 20:01:31 | INFO | M06 | Model artifact will be saved to: artifacts\model.joblib


## Section 2. Load and Prepare the Model

In [2]:
# === Section 2. Load the Data ===

LOG.info(f"Loading dataset: {DATASET_NAME}")
df_model: pd.DataFrame = load_data()
LOG.info(f"Model rows: {df_model.shape[0]}")
LOG.info(f"Classes in '{TARGET_COL}': {sorted(df_model[TARGET_COL].unique())}")

2026-08-07 18:05:37 | INFO | M06 | Loading dataset: penguins
2026-08-07 18:05:37 | INFO | M06 | Loading dataset: penguins
2026-08-07 18:05:37 | INFO | M06 | Loaded: 344 rows, 7 columns
2026-08-07 18:05:38 | INFO | M06 | Model rows (after dropping missing): 333
2026-08-07 18:05:38 | INFO | M06 | Model rows: 333
2026-08-07 18:05:38 | INFO | M06 | Classes in 'sex': ['Female', 'Male']


## Section 3. Split into Train and Test

In [3]:
# === Section 3. Split into Train and Test ===

X_train, X_test, y_train, y_test = split_data(df_model)
LOG.info(f"Train instances: {len(X_train)}")
LOG.info(f"Test instances:  {len(X_test)}")

2026-08-07 18:05:44 | INFO | M06 | Train instances: 266
2026-08-07 18:05:44 | INFO | M06 | Test instances:  67
2026-08-07 18:05:44 | INFO | M06 | Train instances: 266
2026-08-07 18:05:44 | INFO | M06 | Test instances:  67


## Section 4. Train, Save, Reload Model 

In [4]:
# === Section 4. Train, Save, and Reload ===

model = train_model(X_train, y_train)
save_model(model)
model = joblib.load(MODEL_PATH)
LOG.info(f"Reloaded model from: {MODEL_PATH}")

2026-08-07 18:05:49 | INFO | M06 | Training RandomForestClassifier on 266 instances
2026-08-07 18:05:50 | INFO | M06 | Training complete
2026-08-07 18:05:50 | INFO | M06 | Saved model to: artifacts\model.joblib
2026-08-07 18:05:50 | INFO | M06 | Reloaded model from: artifacts\model.joblib


## Section 5. Test the Serving Core

In [5]:
# === Section 5. Test the Serving Core ===

# valid payload - should return a prediction
good_payload: dict[str, Any] = {
    "bill_length_mm": 45.0,
    "bill_depth_mm": 17.0,
    "flipper_length_mm": 200.0,
    "body_mass_g": 4000.0,
}
result: dict[str, Any] = predict_from_features(model, good_payload)
LOG.info(f"Valid payload -> {result}")

# invalid payload - should raise a clean ValueError, not crash
bad_payload: dict[str, Any] = {"bill_length_mm": 45.0}
try:
    predict_from_features(model, bad_payload)
    LOG.warning("Expected a ValueError for the bad payload but none was raised.")
except ValueError as exc:
    LOG.info(f"Invalid payload handled cleanly -> ValueError: {exc}")

c:\Repos\Applied_Machine_Learning\ml-06-serving\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
2026-08-07 18:05:57 | INFO | M06 | Valid payload -> {'prediction': 'Female'}
2026-08-07 18:05:57 | INFO | M06 | Invalid payload handled cleanly -> ValueError: Missing required feature: 'bill_depth_mm'


## Section 6. Summary and Next Steps

First, output key information (may use Python)
Second, provide your narrative, conclusions, and next steps (in Markdown)

In [6]:
# === Section 6. Summary ===

# Python summary
summarize()

2026-08-07 18:07:24 | INFO | M06 | ========================
2026-08-07 18:07:24 | INFO | M06 | SUMMARY
2026-08-07 18:07:24 | INFO | M06 | ========================
2026-08-07 18:07:24 | INFO | M06 | Dataset:  penguins
2026-08-07 18:07:24 | INFO | M06 | Target:   sex
2026-08-07 18:07:24 | INFO | M06 | Features: ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
2026-08-07 18:07:24 | INFO | M06 | Artifact: artifacts\model.joblib
2026-08-07 18:07:24 | INFO | M06 | ========================



### Custom Narrative

In this notebook we trained and tested the data.
Then we tested the server core by pushing through good and bad payloads, along with creating error messages. 

The input contract are 4 characteristics of the penguins; bill length, bill depth, flipper length, and body mass. 

The response gives me the predicted sex or gender of the penguin given the 4 characteristics. It does not give me any probabilities. 

If we wanted to confirm the served model matches the saved one we compare the parameters, like the coefficients, between the freshly trained model and the loaded model.  

### Next Steps

My next step would be to include probability to the responses so that you knew how confident you could be in the prediction. 
